# Résultats : rentabilité de la transition énergétique

Ce notebook **affiche** le tableau final produit par `src/gold/rentabilite.py` (couche gold).
Il ne recalcule rien : il lit, met en forme et dessine.

- en local : lit `data/gold/rentabilite.csv`
- sur Onyxia (`STORAGE=s3`) : lit `s3://<S3_BUCKET>/energie/gold/rentabilite_csv/`

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Racine du projet : le notebook est rangé dans src/
RACINE = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
STORAGE = os.environ.get("STORAGE", "local")

if STORAGE == "s3":
    import s3fs
    fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://" + os.environ.get("AWS_S3_ENDPOINT", "minio.lab.sspcloud.fr")})
    fichier = fs.glob(f"{os.environ['S3_BUCKET']}/energie/gold/rentabilite_csv/*.csv")[0]
    resultats = pd.read_csv(fs.open(fichier), dtype={"departement": str})
else:
    resultats = pd.read_csv(RACINE / "data" / "gold" / "rentabilite.csv", dtype={"departement": str})

resultats

## Années de retour : solution × département

Chaque case = nombre d'années pour que le gain annuel rembourse l'investissement. Vide = jamais rentable.

In [ ]:
tableau = resultats.pivot_table(index="solution", columns=["zone", "nom_departement"],
                                 values="retour_annees")
tableau.round(1)

## Classement dans chaque département

In [ ]:
resultats[resultats["rang"] <= 3][["zone", "nom_departement", "rang", "solution", "retour_annees", "gain_annuel_eur"]]

## Graphique pour le rapport

In [ ]:
departements = resultats[["departement", "nom_departement", "zone"]].drop_duplicates().values
fig, axes = plt.subplots(1, len(departements), figsize=(4.5 * len(departements), 5), sharey=True)

ordre = sorted(resultats["solution"].unique())
for ax, (dept, nom, zone) in zip(axes, departements):
    d = resultats[resultats["departement"] == dept].set_index("solution").reindex(ordre)
    ax.barh(d.index, d["retour_annees"], color="#0f7a5c")
    ax.set_title(f"{nom} ({zone})")
    ax.set_xlabel("années pour se rembourser")
    ax.grid(axis="x", alpha=0.3)
axes[0].invert_yaxis()
fig.suptitle("Temps de retour sur investissement, par solution et par département")
fig.tight_layout()

sortie = RACINE / "data" / "gold" / "graphique_retour.png"
sortie.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(sortie, dpi=150)
print("Graphique enregistré :", sortie)